# 🏋️ Multi-Agent RAG with Microsoft Agent Framework 🥑

Welcome to this fun, self-guided workshop where we'll build a multi-agent Retrieval-Augmented Generation (RAG) system using **Microsoft's native SDKs** with Azure AI Foundry. Our agents will collaborate to answer fitness and health questions in an engaging way!

> **Note:** This demo uses **Microsoft Azure AI frameworks** directly via the `azure-ai-projects`, `azure-ai-inference`, and supporting SDKs — no third-party agent frameworks (AutoGen, Semantic Kernel) required.
>
> ```bash
> pip install azure-ai-projects azure-identity azure-identity-broker azure-ai-inference python-dotenv
> ```

Ensure you've set the following environment variables in your `.env` file:

- `PROJECT_ENDPOINT` (e.g. `https://{hub}.services.ai.azure.com/api/projects/{project}`)
- `AZURE_OPENAI_KEY` (API key for Azure OpenAI inference)
- `MODEL_DEPLOYMENT_NAME` (e.g. `gpt-4o`)

## 1. Setup

Let's import the necessary libraries and set up our model client using Azure AI Foundry. Make sure your environment variables `PROJECT_ENDPOINT` and `MODEL_DEPLOYMENT_NAME` are set (see [.env.example](../../.env.example)) and that you have installed:

```bash
pip install agent-framework agent-framework-foundry agent-framework-orchestrations nest_asyncio
```

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage, AssistantMessage

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

project_endpoint = os.getenv("PROJECT_ENDPOINT", "")
if not project_endpoint:
    raise ValueError("PROJECT_ENDPOINT not set. Please add it to your .env file.")

# Derive base endpoint for inference calls
_parsed       = urlparse(project_endpoint)
base_endpoint = f"{_parsed.scheme}://{_parsed.netloc}"

model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")
api_key               = os.getenv("AZURE_OPENAI_KEY", "")
_deploy_endpoint      = f"{base_endpoint}/openai/deployments/{model_deployment_name}"

# Initialize inference client
use_mock_inference = not bool(api_key.strip())
inference_client   = None

if not use_mock_inference:
    try:
        inference_client = ChatCompletionsClient(
            endpoint=_deploy_endpoint,
            credential=AzureKeyCredential(api_key),
        )
        print("✅ ChatCompletionsClient initialized")
    except Exception as e:
        print(f"⚠️  Could not create ChatCompletionsClient: {e}")
        use_mock_inference = True
else:
    print("⚠️  AZURE_OPENAI_KEY not set — using mock inference for demonstration.")

print(f"✅ Microsoft Agent Framework loaded")
print(f"   Model deployment: {model_deployment_name}")

## 2. Create Sample Health Data & Retrieval Tool

We'll define a knowledge base of health tips and a simple retrieval function that simulates fetching relevant information based on user queries.

In [ ]:
# Define sample health tips knowledge base
health_tips = [
    {"id": "tip1", "content": "Do a 10-minute HIIT workout to boost your metabolism.", "source": "Fitness Guru"},
    {"id": "tip2", "content": "Take a brisk 15-minute walk to clear your mind and improve circulation.", "source": "Health Coach"},
    {"id": "tip3", "content": "Stretch for 5 minutes every hour if you're sitting at a desk.", "source": "Wellness Expert"},
    {"id": "tip4", "content": "Incorporate strength training twice a week for overall fitness.", "source": "Personal Trainer"},
    {"id": "tip5", "content": "Drink water regularly to stay hydrated during workouts.", "source": "Nutritionist"},
    {"id": "tip6", "content": "Get 7-9 hours of sleep every night for better recovery.", "source": "Sleep Specialist"},
    {"id": "tip7", "content": "Do yoga for 20 minutes daily to improve flexibility and reduce stress.", "source": "Yoga Instructor"},
]

def retrieve_health_tips(query: str) -> str:
    """Retrieve relevant health tips from the knowledge base based on the user's query."""
    query_lower = query.lower()
    relevant = []
    
    # Simple keyword matching retrieval
    for tip in health_tips:
        # Check if any word in the query matches content
        if any(word in tip["content"].lower() for word in query_lower.split()):
            relevant.append(f"💡 {tip['source']}: {tip['content']}")
    
    # If no tips match, return all tips (for demo purposes)
    if not relevant:
        relevant = [f"💡 {tip['source']}: {tip['content']}" for tip in health_tips]
    
    return "\n".join(relevant)

print("✅ Health knowledge base & retrieval tool created!")

## 3. Define Multi-Agent RAG System

We'll create a multi-agent system using Microsoft's ChatCompletionsClient:

1. **RetrieverAgent**: Fetches relevant fitness and health tips from the knowledge base
2. **ResponderAgent**: Crafts a fun, engaging response based on retrieved tips

Both agents use the same underlying model but are orchestrated with different system prompts and responsibilities.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class MultiAgentRAGSystem:
    """Multi-agent RAG system using Microsoft ChatCompletionsClient."""
    inference_client: object
    model_name: str
    use_mock: bool = False
    
    def retrieve_and_respond(self, user_query: str) -> dict:
        """Execute the multi-agent RAG pipeline: retrieve then respond."""
        results = {
            'user_query': user_query,
            'retriever_response': '',
            'responder_response': '',
        }
        
        # ── Phase 1: RetrieverAgent fetches relevant tips ────────────────
        print("\n🔍 Phase 1: RetrieverAgent fetching relevant tips...")
        retrieved_context = retrieve_health_tips(user_query)
        results['retriever_response'] = retrieved_context
        
        if self.use_mock:
            responder_response = (
                f"Based on the retrieved tips for '{user_query}', here are my recommendations:\n\n"
                f"{retrieved_context}\n\n"
                "Remember to consult with a healthcare professional before starting any new fitness regimen! 💪"
            )
        else:
            # Use ChatCompletionsClient for retriever context generation
            retriever_messages = [
                SystemMessage(
                    content=(
                        "You are a smart retrieval agent. Your task is to fetch and summarize relevant "
                        "fitness and health tips based on the user's query. Be concise and actionable."
                    )
                ),
                UserMessage(content=f"User question: {user_query}\n\nRetrieved tips:\n{retrieved_context}"),
            ]
            
            retriever_response = self.inference_client.complete(
                model=self.model_name,
                messages=retriever_messages,
            )
            retrieved_context = retriever_response.choices[0].message.content
            results['retriever_response'] = retrieved_context
        
        print(f"RetrieverAgent Response:\n{retrieved_context}")
        
        # ── Phase 2: ResponderAgent crafts engaging response ────────────────
        print("\n💬 Phase 2: ResponderAgent crafting engaging response...")
        
        if self.use_mock:
            final_response = (
                f"Great question about: {user_query}\n\n"
                f"Based on expert advice:\n{retrieved_context}\n\n"
                "Take action today and feel the difference! 🏋️‍♀️✨"
            )
        else:
            responder_messages = [
                SystemMessage(
                    content=(
                        "You are a friendly and motivating fitness coach. Your role is to craft "
                        "fun, engaging, and practical responses to health questions using the provided tips. "
                        "Be encouraging and include actionable advice."
                    )
                ),
                UserMessage(
                    content=(
                        f"User question: {user_query}\n\n"
                        f"Retrieved context from RetrieverAgent:\n{retrieved_context}\n\n"
                        "Please craft a motivating response based on these tips."
                    )
                ),
            ]
            
            responder_response = self.inference_client.complete(
                model=self.model_name,
                messages=responder_messages,
            )
            final_response = responder_response.choices[0].message.content
        
        results['responder_response'] = final_response
        print(f"ResponderAgent (Final Response):\n{final_response}")
        
        return results

# Create the multi-agent RAG system
rag_system = MultiAgentRAGSystem(
    inference_client=inference_client,
    model_name=model_deployment_name,
    use_mock=use_mock_inference,
)

print("✅ Multi-Agent RAG System initialized!")
print("   RetrieverAgent  → fetches relevant health tips from knowledge base")
print("   ResponderAgent  → crafts motivating response from retrieved tips")

## 4. Test the Multi-Agent RAG System

Let's run our multi-agent system with an example fitness query:

> **User Query:** _I'm very busy but want to stay fit. What quick exercises can I do?_

In [ ]:
# Test the multi-agent RAG system with sample queries
test_queries = [
    "I'm very busy but want to stay fit. What quick exercises can I do?",
    "How can I improve my fitness while working at a desk all day?",
]

print("="*70)
print("🚀 MULTI-AGENT RAG SYSTEM DEMO")
print("="*70)

for i, query in enumerate(test_queries, 1):
    print(f"\n\n{'='*70}")
    print(f"Query #{i}: {query}")
    print(f"{'='*70}")
    
    results = rag_system.retrieve_and_respond(query)
    print("\n" + "-"*70)

print("\n✅ Multi-agent RAG demonstration complete!")

## 5. Conclusion

In this notebook, we built a fun multi-agent Retrieval-Augmented Generation pipeline using [Microsoft Agent Framework](https://github.com/microsoft/agent-framework) with Azure AI Foundry and a fitness and health theme. We created a RetrieverAgent that fetches relevant health tips and a ResponderAgent that crafts an engaging answer to the user's query.

Feel free to modify and expand this notebook to explore more advanced multi-agent collaborations (e.g. `ConcurrentBuilder` or `MagenticBuilder` from `agent_framework.orchestrations`). Happy coding and stay fit! 💪🥦